In [1]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
import cv2
import mediapipe as mp
import winsound
import datetime

# -----------------------------
# Setup MediaPipe FaceMesh
# -----------------------------
mp_face_mesh = mp.solutions.face_mesh

face_mesh = mp_face_mesh.FaceMesh(
    max_num_faces=2,
    refine_landmarks=True,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

# -----------------------------
# Camera Setup
# -----------------------------
video = cv2.VideoCapture(0)

previous_status = "Normal"
warnings = 0
MAX_WARNINGS = 5

# -----------------------------
# Main Loop
# -----------------------------
while video.isOpened():
    success, frame = video.read()
    if not success:
        break

    rgb_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb_image)

    face_count = 0
    head_direction = "Straight"
    text_color = (0, 255, 0)

    # -----------------------------
    # Face & Head Direction Detection
    # -----------------------------
    if results.multi_face_landmarks:
        face_count = len(results.multi_face_landmarks)

        for face_landmarks in results.multi_face_landmarks:
            h, w, _ = frame.shape

            nose = face_landmarks.landmark[1]
            left_cheek = face_landmarks.landmark[234]
            right_cheek = face_landmarks.landmark[454]

            nose_x = int(nose.x * w)
            left_x = int(left_cheek.x * w)
            right_x = int(right_cheek.x * w)

            face_center = (left_x + right_x) // 2

            if nose_x < face_center - 15:
                head_direction = "Looking Right"
            elif nose_x > face_center + 15:
                head_direction = "Looking Left"
            else:
                head_direction = "Straight"

            xs = [int(lm.x * w) for lm in face_landmarks.landmark]
            ys = [int(lm.y * h) for lm in face_landmarks.landmark]
            x1, y1 = min(xs), min(ys)
            x2, y2 = max(xs), max(ys)

            cv2.rectangle(frame, (x1, y1), (x2, y2), text_color, 2)

    # -----------------------------
    # Status Logic
    # -----------------------------
    if face_count == 0:
        status = "Face Missing"
        text_color = (0, 0, 255)

    elif face_count > 1:
        status = "Multiple Faces Detected"
        text_color = (0, 0, 255)

    elif head_direction != "Straight":
        status = head_direction
        text_color = (0, 165, 255)

    else:
        status = "Normal"
        text_color = (0, 255, 0)

    # -----------------------------
    # Log + Warning Counter
    # -----------------------------
    if status != previous_status:
        current_time = datetime.datetime.now().strftime("%H:%M:%S")
        print(f"{status} -> {current_time}")

        # Count warnings only for violations
        if status != "Normal":
            warnings += 1
            winsound.Beep(2000, 500)

        previous_status = status

    # -----------------------------
    # Exam Termination Condition
    # -----------------------------
    if warnings > MAX_WARNINGS:
        cv2.putText(
            frame,
            "EXAM TERMINATED!",
            (150, 240),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.5,
            (0, 0, 255),
            4
        )
        cv2.imshow("AI Proctoring System", frame)
        cv2.waitKey(3000)
        break

    # -----------------------------
    # Display Info
    # -----------------------------
    cv2.putText(
        frame,
        f"Faces Detected: {face_count}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        text_color,
        2
    )

    cv2.putText(
        frame,
        f"Status: {status}",
        (20, 80),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        text_color,
        2
    )

    cv2.putText(
        frame,
        f"Warnings: {warnings}/{MAX_WARNINGS}",
        (20, 120),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 0, 255),
        2
    )

    # -----------------------------
    # Show Webcam
    # -----------------------------
    cv2.imshow("AI Proctoring System", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# -----------------------------
# Cleanup
# -----------------------------
video.release()
cv2.destroyAllWindows()


Looking Left -> 15:14:56
Normal -> 15:14:56
Face Missing -> 15:14:56
Normal -> 15:14:57
Looking Right -> 15:14:59
Normal -> 15:15:00
Looking Left -> 15:15:02
